In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror{font-family:Consolas; font-size:15pt;}
div.output{font-size:12pt; font-weight:bold;}
div.input{font-family:Consolas; font-size:12pt;}
div.prompt{min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# <span style="color:red">ch02. LLM활용의 기본 개념(Ollama)</span>

# 1. LLM을 활용하여 답변 생성하기

## 1) Ollama 이용한 로컬 LLM 이용
- 성능은 GPT, Claude 같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ⓐ deepseek-r1:1.5b
- ollama.com 다운로드 -> 설치 -> 모델 pull
- ollama pull deepseek-r1:1.5b (window키 + R => powershell창)
- ollama pull llama3.2:1b

In [4]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='deepseek-r1:1.5b')
result = llm.invoke('What is the capital of Korea?') 
result # 추론모델<think>~<think>

AIMessage(content='<think>\n\n</think>\n\nThe capital of Korea is Hangulji, South Korea.', additional_kwargs={}, response_metadata={'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-25T02:12:18.2614359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1129224000, 'load_duration': 18190300, 'prompt_eval_count': 10, 'prompt_eval_duration': 235603600, 'eval_count': 17, 'eval_duration': 874913900, 'model_name': 'deepseek-r1:1.5b'}, id='run--22c8dd37-f7f2-4358-be35-7547b20412eb-0', usage_metadata={'input_tokens': 10, 'output_tokens': 17, 'total_tokens': 27})

### ⓑ llama3.2:1b
- ollama.com 다운로드 -> 설치 -> 모델 pull
- ollama pull llama3.2:1b (window키 + R => powershell창)
- llama : 공식적으로 한글지원 안 됨(llama3.1 405b는 한글지원 가능 -> llama3.3 70b)
- exaone : 공식적으로 한글지원

In [5]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='llama3.2:1b')
result = llm.invoke('What is the capital of Korea?')
result

AIMessage(content='The capital of South Korea is Seoul.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T02:17:32.1122577Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2302476500, 'load_duration': 1374724900, 'prompt_eval_count': 32, 'prompt_eval_duration': 454721600, 'eval_count': 9, 'eval_duration': 470195600, 'model_name': 'llama3.2:1b'}, id='run--32035115-6bf6-4a7c-9549-46beffc7db06-0', usage_metadata={'input_tokens': 32, 'output_tokens': 9, 'total_tokens': 41})

In [6]:
result.content

'The capital of South Korea is Seoul.'

In [10]:
result = llm.invoke('한국의 수도는 어디예요?')
result.content

'한국의 수도는 Seoul입니다.'

## 2) openai 활용
- pip install langchain-openai

In [12]:
from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model='gpt-4o-mini')
# result = llm.invoke('What is the capital of Korea?')
# result => 에러 이유 : OPENAI_API_KEY 환경변수 부재

In [16]:
# 환경 변수 가져오기
from dotenv import load_dotenv
# import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

True

In [ ]:
# 코랩에서 OPENAI_API_KEY 읽어오기(.env 사용 불가능)
# 보안키 추가 후 아래 소스 실행
# from google.colab import userdata
# userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano', 
                 # openai_api_key=os.getenv('OPENAI_API_KEY')
                )
# llm.invoke("What is the capital of Korea? Answer me in Korean")

In [ ]:
# 모든 모델의 키가 OPENAI_API_KEY는 아님(에러 메세지로 키 확인 가능, 안될 때도 있음)
# Claude -> Anthropic
# Azure, upstage, Bedrock : 에러 메세지 참조하여 환경변수 생성

In [ ]:
# from langchain_openai import AzureOpenAI
# llm = AzureOpenAI(model='gpt-4.1-nano')
# llm.invoke('What is the capital of Korea?')
# 에러를 내면 -> OPENAI_API_VERSION 환경변수가 필요하다는 메세지를 볼 수 있음

In [ ]:
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model='claude-3.5-sonnet-20240620')
# llm.invoke('What is the capital of Korea?')
# Claude는 에러 메세지를 봐도 환경변수 이름을 알 수 없음 
# -> ChatAnthropic 검색 후, langchain docs 사이트로 이동
# -> docs에서 명시한 'ANTHROPIC_API_KEY' 이름의 환경변수 설정

# 2. 렝체인 스타일로 프롬프트(질문) 작성하기
- 프롬프트 : llm 호출 시 invoke 안에 쓰는 질문

In [18]:
llm = ChatOllama(model='llama3.2:1b')
# llm.invoke(0) => 에러남
# 프롬프트 타입 : 스트링, PromptValue, BaseMessage리스트

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate을 사용하여 변수가 포함된 템플릿 작성

In [20]:
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model='llama3.2:1b')
prompt_template = PromptTemplate(
    # 아래 두개는 PromptValue 
    template = "What is the capital of {country}", # {}안의 값을 새로운 값으로 대입 가능
    input_variables = ["country"]
)
prompt = prompt_template.invoke({"country":"Korea"})
print(prompt)
llm.invoke(prompt)

text='What is the capital of Korea'


AIMessage(content="The capital of South Korea is Seoul. However, it's worth noting that there are also two other cities that serve as temporary capitals during major events or construction projects: Busan and Daejeon. But Seoul has been the official capital since 1948.", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T05:31:46.7999563Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3217699800, 'load_duration': 18742900, 'prompt_eval_count': 31, 'prompt_eval_duration': 61564800, 'eval_count': 54, 'eval_duration': 3136673400, 'model_name': 'llama3.2:1b'}, id='run--b77b0947-9f5d-46a4-9821-4139ee3b8e44-0', usage_metadata={'input_tokens': 31, 'output_tokens': 54, 'total_tokens': 85})

## 2) 메세지 기반 프롬프트 작성
- BaseMessage리스트
- BaseMessage 상속 받은 클래스 : AIMessage, HummanMessage, SystemMessage, ToolMessage

In [21]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant!"),
    HumanMessage(content="What is the capital of Italy?"), # 질문
    AIMessage(content="The capital of Italy is Rome."), # 답
    HumanMessage(content="What is the capital of Korea?"),
    AIMessage(content="The capital of Italy is Seoul."),
    HumanMessage(content="What is the capital of France?")
]
llm.invoke(message_list)

AIMessage(content="You're probably thinking of a different country. The capital of France is Paris.", additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T05:51:07.8039197Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5082676800, 'load_duration': 2291322600, 'prompt_eval_count': 86, 'prompt_eval_duration': 1743413300, 'eval_count': 17, 'eval_duration': 1045089200, 'model_name': 'llama3.2:1b'}, id='run--4bbf9e28-cef3-455d-bb96-573b614b2530-0', usage_metadata={'input_tokens': 86, 'output_tokens': 17, 'total_tokens': 103})

In [23]:
# BaseMessage list로 하면 렝체인화 X, ChatPromptTemplate X
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
llm = ChatOllama(model='llama3.2:1b')
message_list = [
    SystemMessage(content="You are a helpful assistant!"),
    HumanMessage(content="What is the capital of Italy?"), 
    AIMessage(content="The capital of Italy is Rome."), 
    HumanMessage(content="What is the capital of Korea?"),
    AIMessage(content="The capital of Italy is Seoul."),
    HumanMessage(content="What is the capital of {country}?")
]
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate.from_messages(message_list)
prompt = chatPromptTemplate.invoke({'country':'Korea'})
print(prompt)

messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Italy?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Rome.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of Korea?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of Italy is Seoul.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of {country}?', additional_kwargs={}, response_metadata={})]


## 3) ChatPromptTemplate 사용
- BaseMessage 리스트 -> 튜플 리스트

In [27]:
# 위의 BaseMessage를 수정
chatPromptTemplate = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant!"),
    ("human", "What is the capital of {country}?")
])
country = input('어느 나라 수도가 궁금하세요?')
prompt = chatPromptTemplate.invoke({'country':country})
print("프롬프트 :", prompt)
result = llm.invoke(prompt)
result.content

어느 나라 수도가 궁금하세요?한국
프롬프트 : messages=[SystemMessage(content='You are a helpful assistant!', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of 한국?', additional_kwargs={}, response_metadata={})]


'The capital of South Korea is Seoul.'

In [30]:
chatPromptTemplate = ChatPromptTemplate.from_messages([
    ("system", "당신은 대한민국 전문 도우미야!"),
    ("human", "{country}의 수도가 어디예요?")
])
country = input('어느 나라 수도가 궁금하세요?')
prompt = chatPromptTemplate.invoke({'country':country})
print("프롬프트 :", prompt)
result = llm.invoke(prompt)
result.content

어느 나라 수도가 궁금하세요?한국
프롬프트 : messages=[SystemMessage(content='당신은 대한민국 전문 도우미야!', additional_kwargs={}, response_metadata={}), HumanMessage(content='한국의 수도가 어디예요?', additional_kwargs={}, response_metadata={})]


'한국의 수도는 Seoul입니다.'

# 3. 답변 형식을 컨트롤하기
- invoke 실행결과는 AIMessage() -> String이나 Json, 객체 : outputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 사용하여 LLM 출력(AIMessage)을 단순 문자열로 변환

In [35]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
# 명시적인 지시사항이 포함된 프롬프트 생성
prompt_template = PromptTemplate(
    template = "What is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({"country":"Korea"})
print("프롬프트 :", prompt)
result = llm.invoke(prompt)
print("llm 결과 :", type(result), result)
# 문자열 출력 파서를 이용하여 llm응답을 단순 문자열 변환
output_parser = StrOutputParser()
print("파서 결과 :", output_parser.invoke(result))

프롬프트 : text='What is the capital of Korea. Return the name of the city only'
llm 결과 : <class 'langchain_core.messages.ai.AIMessage'> content='Seoul' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T06:39:33.3016244Z', 'done': True, 'done_reason': 'stop', 'total_duration': 206918800, 'load_duration': 27906000, 'prompt_eval_count': 39, 'prompt_eval_duration': 61305300, 'eval_count': 3, 'eval_duration': 117321900, 'model_name': 'llama3.2:1b'} id='run--ff10f450-296f-4e3d-bb00-239cf684a8ef-0' usage_metadata={'input_tokens': 39, 'output_tokens': 3, 'total_tokens': 42}
파서 결과 : Seoul


In [37]:
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"korea"})))

'Seoul'

In [39]:
# PromptTemplate(변수설정) => ChatPromptTemplate(변수설정, system과 모범답안 지정)
llm = ChatOllama(model='llama3.2:1b')

chat_prompt_template = ChatPromptTemplate([ # ChatPromptTemplate은 리스트로
    ("system", "You are a helpful assistant with expertise in South Korea."),
    ("human", "What is the capital of {country}? Return the name if the city only.")
])

output_parser = StrOutputParser()

output_parser.invoke(llm.invoke(chat_prompt_template.invoke({"country":"Korea"})))

'Seoul'

## 2) Json 출력 파서 이용
- json()으로 응답하기를 원하지만, 우선 어떤 형식으로 반환되는 지 확인
- {"name":"홍","age":20}(json) / {'name':'홍','age':20}(dict)

In [45]:
from langchain_core.output_parsers import JsonOutputParser
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
prompt = country_detail_prompt.invoke({"country":"Korea"})
print(type(prompt), prompt)
# Json output 파서
output_parser = JsonOutputParser()
ai_message = llm.invoke(prompt)
print(type(prompt), ai_message)
json_result = output_parser.invoke(ai_message)
print(type(json_result))

<class 'langchain_core.prompt_values.StringPromptValue'> text='Give following information about Korea\n    - Capital\n    - Population\n    - Language\n    - Currency\n    return it is JSON format and return the JSON dictionary only'
<class 'langchain_core.prompt_values.StringPromptValue'> content='```\n{\n  "capital": "Seoul",\n  "population": 51000000,\n  "language": "Korean",\n  "currency": "KRW"\n}\n```' additional_kwargs={} response_metadata={'model': 'llama3.2:1b', 'created_at': '2025-06-25T07:01:51.6304143Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2313421200, 'load_duration': 28712200, 'prompt_eval_count': 59, 'prompt_eval_duration': 61651900, 'eval_count': 39, 'eval_duration': 2222541800, 'model_name': 'llama3.2:1b'} id='run--37767b08-aab6-40cd-adbb-59a341b59532-0' usage_metadata={'input_tokens': 59, 'output_tokens': 39, 'total_tokens': 98}
<class 'dict'>


In [47]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
output_parser = JsonOutputParser()
info = output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))
info

{'capital': 'Seoul',
 'population': 51000000,
 'language': 'Korean',
 'currency': 'Korean won'}

In [48]:
type(info)

dict

## 3) 구조화된 출력 사용
- Pydantic 모델을 사용하여 LLM 출력을 구조화된 형식으로 받기(JsonParser보다 훨씬 안정적)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리

In [55]:
# 일반적인
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
#     def __str__(self):
#         return self.name + " " + self.id
user = User(1, "홍길동")
print(user)

In [58]:
from pydantic import BaseModel, Field
class User(BaseModel):
    # gt=0:id>0, ge=0:id>=0, lt=0:id<0, le=0:id<=0
    id:int = Field(gt=0, description="id") 
    name:str = Field(min_length=2, description="name")
    is_active:bool = Field(default=True, description="id활성화")
user = User(id="1", name="홍길동") # id="1" 자동적으로 1로 형변환 해줌
print(user)

id=1 name='홍길동' is_active=True


In [61]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]
)
class CountryDetail(BaseModel): # description : 더 정확한 출력 유도
    capital:str    = Field(description="the capital of the country")
    population:int = Field(description="the population of the country")
    language:str   = Field(description="the language of the country")
    currency:str   = Field(description="the currency of the country")
# 출력 형식 파서 + LLM
structedllm = llm.with_structured_output(CountryDetail)

# 아래는 기존의 방식
# output_parser = JsonOutputParser()
# output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))

info = structedllm.invoke(country_detail_prompt.invoke({"country":"Korea"}))
type(info) # __main__.CountryDetail : 객체로 받음

__main__.CountryDetail

In [63]:
print(info)
print(info.capital, info.population, info.language, info.currency)

capital='Seoul' population=51000000 language='Korean' currency='Won'
Seoul 51000000 Korean Won


In [67]:
print('info를 json :', info.model_dump_json()) # model_dump_json() : json() 함수와 동일
print('info를 dict :', info.model_dump()) # model_dump : dict()

info를 json : {"capital":"Seoul","population":51000000,"language":"Korean","currency":"Won"}
info를 dict : {'capital': 'Seoul', 'population': 51000000, 'language': 'Korean', 'currency': 'Won'}


# 4. LCEL을 활용한 렝체인 생성하기
## 1) 문자열 출력 파서 사용
- invoke

In [68]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model="llama3.2:1b",
                 temperature=0) # 일관된 답변
# 명시적인 지시사항이 포함된 프롬프트 생성
prompt_template = PromptTemplate(
    template = "What is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Korea"})))

'Seoul'

## 2) LCEL을 사용한 간단한 체인 구성
- invoke를 사용하지 않고 파이프연산자(|) 사용

In [69]:
# 프롬프트템플릿 -> llm -> 출력 파서를 연결하는 체인 생성
capital_chain = prompt_template | llm | output_parser
# 생성된 체인을 invoke
capital_chain.invoke({"country":"Korea"})

'Seoul'